In [5]:
import os
import io
import json
import time
import tarfile
import zipfile
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_STORAGE = Path("/content/drive/MyDrive/SHIELD_DATA/raw_store/benign")
NPM_STORAGE = BASE_STORAGE / "npm"
PYPI_STORAGE = BASE_STORAGE / "pypi"

NPM_STORAGE.mkdir(parents=True, exist_ok=True)
PYPI_STORAGE.mkdir(parents=True, exist_ok=True)

# Files critical for Tier-1 Metadata & Tier-2 AST Chunking
TARGET_FILES = {
    "package.json", "setup.py", "setup.cfg", "pyproject.toml",
    "preinstall.js", "postinstall.js", "index.js", "__init__.py"
}

def extract_selective(archive_bytes: bytes, target_dir: Path, is_zip=False):
    """Extracts only critical entry points and scripts to conserve space."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if is_zip:
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
            for member in z.namelist():
                base_name = os.path.basename(member)
                if base_name in TARGET_FILES:
                    out_path = target_dir / base_name
                    with open(out_path, "wb") as f:
                        f.write(z.read(member))
    else:
        with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:*") as tar:
            for member in tar.getmembers():
                base_name = os.path.basename(member.name)
                if base_name in TARGET_FILES and member.isfile():
                    f = tar.extractfile(member)
                    if f:
                        out_path = target_dir / base_name
                        with open(out_path, "wb") as out:
                            out.write(f.read())

def fetch_npm_package(session: requests.Session, pkg_name: str):
    """Fetches npm latest package metadata and unpacks build hooks & entrypoint."""
    meta_url = f"https://registry.npmjs.org/{pkg_name}"
    try:
        r = session.get(meta_url, timeout=12)
        if r.status_code != 200:
            return False, pkg_name, f"HTTP {r.status_code}"

        doc = r.json()
        latest_ver = doc.get("dist-tags", {}).get("latest")
        if not latest_ver or latest_ver not in doc.get("versions", {}):
            return False, pkg_name, "No latest release"

        dist = doc["versions"][latest_ver].get("dist", {})
        tarball_url = dist.get("tarball")
        if not tarball_url:
            return False, pkg_name, "No tarball URL"

        tar_resp = session.get(tarball_url, timeout=20)
        if tar_resp.status_code == 200:
            target_path = NPM_STORAGE / pkg_name / latest_ver
            extract_selective(tar_resp.content, target_path, is_zip=False)
            return True, pkg_name, "Success"
        return False, pkg_name, f"Download failed: {tar_resp.status_code}"
    except Exception as e:
        return False, pkg_name, str(e)

def fetch_pypi_package(session: requests.Session, pkg_name: str):
    """Fetches PyPI sdist package to guarantee preservation of setup.py."""
    meta_url = f"https://pypi.org/pypi/{pkg_name}/json"
    try:
        r = session.get(meta_url, timeout=12)
        if r.status_code != 200:
            return False, pkg_name, f"HTTP {r.status_code}"

        doc = r.json()
        version = doc.get("info", {}).get("version")
        releases = doc.get("releases", {}).get(version, [])

        # Prioritize source distributions (.tar.gz / .zip) over .whl
        sdist = next((rel for rel in releases if rel.get("packagetype") == "sdist"), None)
        is_zip = False
        if not sdist and releases:
            sdist = releases[0]
            is_zip = sdist["filename"].endswith((".zip", ".whl"))
        if not sdist:
            return False, pkg_name, "No release files found"

        file_url = sdist.get("url")
        file_resp = session.get(file_url, timeout=20)
        if file_resp.status_code == 200:
            target_path = PYPI_STORAGE / pkg_name / version
            extract_selective(file_resp.content, target_path, is_zip=is_zip)
            return True, pkg_name, "Success"
        return False, pkg_name, f"Download failed: {file_resp.status_code}"
    except Exception as e:
        return False, pkg_name, str(e)

def execute_ingestion_worker(ecosystem: str, workers=16):
    seed_file = Path(f"data/seed_lists/{ecosystem}_benign_seeds.json")
    if not seed_file.exists():
        raise FileNotFoundError(f"Missing {seed_file}. Run generate_pull_lists.py first.")

    with open(seed_file, "r") as f:
        pkg_list = json.load(f)

    print(f"[*] Starting ingestion for {len(pkg_list)} {ecosystem} packages with {workers} threads...")

    session = requests.Session()
    # Adapter connection pooling prevents port exhaustion
    adapter = requests.adapters.HTTPAdapter(pool_connections=workers, pool_maxsize=workers * 2, max_retries=2)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    fetch_fn = fetch_npm_package if ecosystem == "npm" else fetch_pypi_package

    success, fail = 0, 0
    start_time = time.time()

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(fetch_fn, session, name): name for name in pkg_list}
        for future in as_completed(futures):
            ok, name, status = future.result()
            if ok:
                success += 1
            else:
                fail += 1
            if (success + fail) % 250 == 0:
                elapsed = time.time() - start_time
                print(f"[{ecosystem.upper()}] Processed: {success + fail}/{len(pkg_list)} | Success: {success} | Failed: {fail} | Speed: {(success + fail) / elapsed:.1f} pkg/s")

    print(f"[✓] {ecosystem.upper()} complete. Successfully preserved: {success}, Skipped/Failed: {fail}")

if __name__ == "__main__":
    execute_ingestion_worker("npm", workers=12)
    execute_ingestion_worker("pypi", workers=12)

FileNotFoundError: Missing data/seed_lists/npm_benign_seeds.json. Run generate_pull_lists.py first.

In [ ]:
import os
import io
import json
import time
import tarfile
import zipfile
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Define isolated Google Drive storage paths
BASE_STORAGE = Path("/content/drive/MyDrive/SHIELD_DATA/raw_store/benign")
NPM_STORAGE = BASE_STORAGE / "npm"
PYPI_STORAGE = BASE_STORAGE / "pypi"

NPM_STORAGE.mkdir(parents=True, exist_ok=True)
PYPI_STORAGE.mkdir(parents=True, exist_ok=True)

# Files critical for Tier-1 Metadata & Tier-2 AST Chunking
TARGET_FILES = {
    "package.json", "setup.py", "setup.cfg", "pyproject.toml",
    "preinstall.js", "postinstall.js", "index.js", "__init__.py"
}

def extract_selective(archive_bytes: bytes, target_dir: Path, is_zip=False):
    """Extracts only critical entry points and scripts to conserve space."""
    target_dir.mkdir(parents=True, exist_ok=True)
    if is_zip:
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
            for member in z.namelist():
                base_name = os.path.basename(member)
                if base_name in TARGET_FILES:
                    out_path = target_dir / base_name
                    with open(out_path, "wb") as f:
                        f.write(z.read(member))
    else:
        with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:*") as tar:
            for member in tar.getmembers():
                base_name = os.path.basename(member.name)
                if base_name in TARGET_FILES and member.isfile():
                    f = tar.extractfile(member)
                    if f:
                        out_path = target_dir / base_name
                        with open(out_path, "wb") as out:
                            out.write(f.read())

def fetch_npm_package(session: requests.Session, pkg_name: str):
    """Fetches npm latest package metadata and unpacks build hooks."""
    meta_url = f"https://registry.npmjs.org/{pkg_name}"
    try:
        r = session.get(meta_url, timeout=12)
        if r.status_code != 200:
            return False, pkg_name, f"HTTP {r.status_code}"

        doc = r.json()
        latest_ver = doc.get("dist-tags", {}).get("latest")
        if not latest_ver or latest_ver not in doc.get("versions", {}):
            return False, pkg_name, "No latest release"

        dist = doc["versions"][latest_ver].get("dist", {})
        tarball_url = dist.get("tarball")
        if not tarball_url:
            return False, pkg_name, "No tarball URL"

        tar_resp = session.get(tarball_url, timeout=20)
        if tar_resp.status_code == 200:
            target_path = NPM_STORAGE / pkg_name / latest_ver
            extract_selective(tar_resp.content, target_path, is_zip=False)
            return True, pkg_name, "Success"
        return False, pkg_name, f"Download failed: {tar_resp.status_code}"
    except Exception as e:
        return False, pkg_name, str(e)

def fetch_pypi_package(session: requests.Session, pkg_name: str):
    """Fetches PyPI sdist package to guarantee preservation of setup.py."""
    meta_url = f"https://pypi.org/pypi/{pkg_name}/json"
    try:
        r = session.get(meta_url, timeout=12)
        if r.status_code != 200:
            return False, pkg_name, f"HTTP {r.status_code}"

        doc = r.json()
        version = doc.get("info", {}).get("version")
        releases = doc.get("releases", {}).get(version, [])

        # Prioritize source distributions (.tar.gz / .zip) over .whl
        sdist = next((rel for rel in releases if rel.get("packagetype") == "sdist"), None)
        is_zip = False
        if not sdist and releases:
            sdist = releases[0]
            is_zip = sdist["filename"].endswith((".zip", ".whl"))
        if not sdist:
            return False, pkg_name, "No release files found"

        file_url = sdist.get("url")
        file_resp = session.get(file_url, timeout=20)
        if file_resp.status_code == 200:
            target_path = PYPI_STORAGE / pkg_name / version
            extract_selective(file_resp.content, target_path, is_zip=is_zip)
            return True, pkg_name, "Success"
        return False, pkg_name, f"Download failed: {file_resp.status_code}"
    except Exception as e:
        return False, pkg_name, str(e)

def execute_ingestion_worker(ecosystem: str, workers=16):
    seed_file = Path(f"data/seed_lists/{ecosystem}_benign_seeds.json") # Corrected path
    if not seed_file.exists():
        raise FileNotFoundError(f"Missing {seed_file}. Please ensure Day 2 lists are generated.")

    with open(seed_file, "r") as f:
        pkg_list = json.load(f)

    print(f"[*] Starting ingestion for {len(pkg_list)} {ecosystem} packages with {workers} threads...")

    session = requests.Session()
    # Adapter connection pooling prevents port exhaustion
    adapter = requests.adapters.HTTPAdapter(pool_connections=workers, pool_maxsize=workers * 2, max_retries=3)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    fetch_fn = fetch_npm_package if ecosystem == "npm" else fetch_pypi_package

    success, fail = 0, 0
    failed_packages = []
    start_time = time.time()

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(fetch_fn, session, name): name for name in pkg_list}
        for future in as_completed(futures):
            ok, name, status = future.result()
            if ok:
                success += 1
            else:
                fail += 1
                failed_packages.append({"name": name, "error": status})

            if (success + fail) % 250 == 0:
                elapsed = time.time() - start_time
                print(f"[{ecosystem.upper()}] Processed: {success + fail}/{len(pkg_list)} | Success: {success} | Failed: {fail} | Speed: {(success + fail) / elapsed:.1f} pkg/s")

    # Log failures for retry loops
    if failed_packages:
        fail_log = BASE_STORAGE / f"{ecosystem}_failed_ingestion.json"
        with open(fail_log, "w") as f:
            json.dump(failed_packages, f, indent=2)
        print(f"[!] Saved {len(failed_packages)} failures to {fail_log}")

    print(f"[✓] {ecosystem.upper()} complete. Successfully preserved: {success}, Skipped/Failed: {fail}")

if __name__ == "__main__":
    # 12-16 workers keeps throughput high without hitting API rate limits
    execute_ingestion_worker("npm", workers=12)
    execute_ingestion_worker("pypi", workers=12)

[*] Starting ingestion for 1691 npm packages with 12 threads...
[NPM] Processed: 250/1691 | Success: 250 | Failed: 0 | Speed: 2.7 pkg/s
[NPM] Processed: 500/1691 | Success: 500 | Failed: 0 | Speed: 2.6 pkg/s
[NPM] Processed: 750/1691 | Success: 750 | Failed: 0 | Speed: 2.8 pkg/s
[NPM] Processed: 1000/1691 | Success: 998 | Failed: 2 | Speed: 2.9 pkg/s
[NPM] Processed: 1250/1691 | Success: 1248 | Failed: 2 | Speed: 3.0 pkg/s
[NPM] Processed: 1500/1691 | Success: 1498 | Failed: 2 | Speed: 3.0 pkg/s
[!] Saved 2 failures to /content/drive/MyDrive/SHIELD_DATA/raw_store/benign/npm_failed_ingestion.json
[✓] NPM complete. Successfully preserved: 1689, Skipped/Failed: 2
[*] Starting ingestion for 3500 pypi packages with 12 threads...
[PYPI] Processed: 250/3500 | Success: 247 | Failed: 3 | Speed: 1.7 pkg/s
[PYPI] Processed: 500/3500 | Success: 477 | Failed: 23 | Speed: 1.7 pkg/s
[PYPI] Processed: 750/3500 | Success: 727 | Failed: 23 | Speed: 1.9 pkg/s
[PYPI] Processed: 1000/3500 | Success: 977 | 

In [ ]:
import os
import hashlib
import tarfile
import pandas as pd
import zstandard as zstd
from pathlib import Path

# Configuration
DATA_ROOT = Path("/content/drive/MyDrive/SHIELD_DATA/raw_store")
SHARD_DIR = Path("/content/drive/MyDrive/SHIELD_DATA/shards")
MANIFEST_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest.parquet")

SHARD_DIR.mkdir(parents=True, exist_ok=True)
MAX_SHARD_SIZE = 2 * 1024**3  # 2 GB per shard

class ShardedTarWriter:
    def __init__(self, out_dir, prefix="shield_shard"):
        self.out_dir = out_dir
        self.prefix = prefix
        self.shard_idx = 0
        self.current_size = 0
        self._open_new_shard()

    def _open_new_shard(self):
        self.shard_idx += 1
        self.shard_name = f"{self.prefix}_{self.shard_idx:03d}.tar.zst"
        self.file_path = self.out_dir / self.shard_name
        self.fp = open(self.file_path, "wb")

        # Level 3 provides an optimal balance between fast writes and strong compression
        self.cctx = zstd.ZstdCompressor(level=3, threads=-1)
        self.stream = self.cctx.stream_writer(self.fp)
        self.tar = tarfile.open(fileobj=self.stream, mode="w|")
        self.current_size = 0
        print(f"[*] Opened new shard: {self.shard_name}")

    def add_directory(self, source_dir, arcname):
        """Adds a directory to the tar and tracks approximate uncompressed size."""
        # Calculate raw file sizes (excludes 512-byte tar header overhead)
        dir_size = sum(f.stat().st_size for f in Path(source_dir).rglob('*') if f.is_file())

        if self.current_size + dir_size > MAX_SHARD_SIZE and self.current_size > 0:
            self.close()
            self._open_new_shard()

        start_offset = self.current_size
        self.tar.add(source_dir, arcname=arcname)
        self.current_size += dir_size

        return self.shard_name, start_offset, dir_size

    def close(self):
        if hasattr(self, 'tar') and self.tar:
            self.tar.close()
            self.stream.close()
            self.fp.close()

def compile_dataset():
    records = []
    writer = ShardedTarWriter(SHARD_DIR)

    for label_category in ["malicious", "benign"]:
        base_dir = DATA_ROOT / label_category
        if not base_dir.exists():
            continue

        for source_collection in base_dir.iterdir():
            if not source_collection.is_dir():
                continue

            for root, dirs, files in os.walk(source_collection):
                # Identify package root by the presence of metadata files
                if any(f in files for f in ["package.json", "setup.py", "pyproject.toml", "setup.cfg"]):
                    pkg_path = Path(root)
                    rel_path = pkg_path.relative_to(DATA_ROOT).as_posix()
                    ecosystem = "npm" if "package.json" in files else "pypi"

                    uid = hashlib.sha256(rel_path.encode()).hexdigest()[:16]

                    # Pack into shard and get stream coordinates
                    shard_id, offset, size_bytes = writer.add_directory(pkg_path, arcname=rel_path)

                    records.append({
                        "uid": uid,
                        "ecosystem": ecosystem,
                        "label": 1 if label_category == "malicious" else 0,
                        "source_collection": source_collection.name,
                        "package_rel_path": rel_path,
                        "shard_id": shard_id,
                        "uncompressed_offset": offset,
                        "size_bytes": size_bytes,
                        "has_install_script": ("setup.py" in files or "package.json" in files)
                    })

                    # Prevent deep traversal into subdirectories of the current package
                    dirs.clear()

    writer.close()

    # Save the unified manifest
    df = pd.DataFrame(records)
    # Ensure no duplicates from overlapping vulnerability collections
    df.drop_duplicates(subset=["package_rel_path"], inplace=True)
    df.to_parquet(MANIFEST_PATH, engine="pyarrow", compression="snappy")

    print(f"[+] Dataset compiled into {writer.shard_idx} shards.")
    print(f"[+] Unified manifest with {len(df)} packages saved to {MANIFEST_PATH}")

if __name__ == "__main__":
    compile_dataset()

[*] Opened new shard: shield_shard_001.tar.zst
[+] Dataset compiled into 1 shards.
[+] Unified manifest with 5284 packages saved to /content/drive/MyDrive/SHIELD_DATA/manifest.parquet


In [ ]:
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/SHIELD_DATA/manifest.parquet")

print(f"Total Packages: {len(df)}")
print(df.groupby(['ecosystem', 'label']).size().unstack(fill_value=0))

corrupt = df[df['size_bytes'] == 0]
if not corrupt.empty:
    print(f"[!] Warning: {len(corrupt)} packages have 0 bytes and failed to pack.")
else:
    print("[✓] All packages successfully packed and indexed by byte-offset.")

Total Packages: 5284
label         0
ecosystem      
npm        1807
pypi       3477
[✓] All packages successfully packed and indexed by byte-offset.


In [ ]:
pip install jellyfish pyarrow pandas zstandard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 13.0 MB/s eta 0:00:00


In [ ]:
import os
import ast
import json
import re
import tarfile
import io
import pandas as pd
import jellyfish
import zstandard as zstd
from pathlib import Path

# --- Storage Paths ---
DATA_ROOT = Path("/content/drive/MyDrive/SHIELD_DATA/raw_store")
SHARDS_DIR = Path("/content/drive/MyDrive/SHIELD_DATA/shards")
MANIFEST_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest.parquet")
OUTPUT_FEATURES = Path("/content/drive/MyDrive/SHIELD_DATA/tier1_features.parquet")

# --- Top 100 Canonical Targets for Typosquat Distance ---
TOP_NPM = [
    "lodash", "chalk", "react", "express", "commander", "moment", "axios",
    "tslib", "debug", "vue", "request", "async", "prop-types", "fs-extra",
    "body-parser", "bluebird", "uuid", "yargs", "glob", "rxjs", "webpack",
    "mkdirp", "minimist", "underscore", "semver", "core-js", "dotenv",
    "colors", "inherits", "inquirer", "chokidar", "classnames", "superagent",
    "source-map", "babel-core", "postcss", "styled-components", "redis",
    "prettier", "mocha", "eslint", "cheerio", "ws", "path-to-regexp", "rimraf"
]

TOP_PYPI = [
    "urllib3", "six", "botocore", "requests", "certifi", "boto3", "setuptools",
    "python-dateutil", "idna", "charset-normalizer", "s3transfer", "typing-extensions",
    "pip", "wheel", "numpy", "cffi", "pydantic", "pytz", "cryptography", "jinja2",
    "pandas", "attrs", "click", "jmespath", "pyyaml", "packaging", "markupsafe",
    "pluggy", "pytest", "scipy", "virtualenv", "importlib-metadata", "aiohttp",
    "psutil", "tomli", "google-api-python-client", "grpcio", "tqdm", "protobuf"
]

# --- Lexical Distance Calculators ---
def compute_typosquat_metrics(name: str, target_list: list):
    clean_name = name.lower().replace("@", "").split("/")[-1]
    min_lev = 999
    max_jw = 0.0

    for target in target_list:
        lev = jellyfish.levenshtein_distance(clean_name, target)
        jw = jellyfish.jaro_winkler_similarity(clean_name, target)
        if lev < min_lev:
            min_lev = lev
        if jw > max_jw:
            max_jw = jw

    return min_lev, max_jw

# --- Static AST Parser for Python setup.py ---
def parse_setup_py_ast(code_str: str):
    """Statically parses setup.py to detect cmdclass and install requires without execution."""
    features = {
        "author_count": 0,
        "desc_len": 0,
        "has_homepage": 0,
        "dep_count": 0,
        "has_cmdclass": 0
    }

    # Regex fallback for high-risk custom commands if AST fails
    if re.search(r"cmdclass\s*=\s*\{", code_str):
        features["has_cmdclass"] = 1

    try:
        tree = ast.parse(code_str)
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                func_name = getattr(node.func, "id", None) or getattr(node.func, "attr", None)
                if func_name == "setup":
                    for keyword in node.keywords:
                        k = keyword.arg
                        # ETM: Description length
                        if k == "description" and isinstance(keyword.value, ast.Constant):
                            features["desc_len"] = len(str(keyword.value.value))
                        # ETM: Homepage / Repositories
                        elif k in ["url", "project_urls"]:
                            features["has_homepage"] = 1
                        # ETM: Author count
                        elif k in ["author", "maintainer"]:
                            features["author_count"] += 1
                        # DTM: Dependencies
                        elif k == "install_requires" and isinstance(keyword.value, (ast.List, ast.Tuple)):
                            features["dep_count"] = len(keyword.value.elts)
                        # Lifecycle Hooks: cmdclass override
                        elif k == "cmdclass":
                            features["has_cmdclass"] = 1
    except Exception:
        # Fallback to regex heuristic on syntax error
        pass

    return features

# --- Parser for npm package.json ---
def parse_package_json(content_str: str):
    features = {
        "desc_len": 0,
        "has_readme": 0,
        "author_count": 0,
        "has_homepage": 0,
        "dep_count": 0,
        "dev_dep_count": 0,
        "has_preinstall": 0,
        "has_postinstall": 0
    }
    try:
        data = json.loads(content_str)
        desc = data.get("description", "")
        features["desc_len"] = len(desc) if isinstance(desc, str) else 0
        features["has_readme"] = 1 if data.get("readme") else 0

        # Authors & maintainers count
        authors = data.get("author") or data.get("contributors") or data.get("maintainers") or []
        if isinstance(authors, list):
            features["author_count"] = len(authors)
        elif isinstance(authors, (dict, str)):
            features["author_count"] = 1

        features["has_homepage"] = 1 if data.get("homepage") or data.get("repository") else 0
        features["dep_count"] = len(data.get("dependencies", {}))
        features["dev_dep_count"] = len(data.get("devDependencies", {}))

        # Lifecycle Hooks Check
        scripts = data.get("scripts", {})
        if isinstance(scripts, dict):
            if "preinstall" in scripts:
                features["has_preinstall"] = 1
            if "postinstall" in scripts or "install" in scripts:
                features["has_postinstall"] = 1
    except Exception:
        pass

    return features

# --- Main Ingestion Loop ---
def extract_metadata_features():
    df_manifest = pd.read_parquet(MANIFEST_PATH)
    print(f"[*] Extracting Tier-1 features across {len(df_manifest)} indexed packages...")

    feature_rows = []

    for _, row in df_manifest.iterrows():
        uid = row["uid"]
        ecosystem = row["ecosystem"]
        label = row["label"]
        rel_path = row["package_rel_path"]
        pkg_full_path = DATA_ROOT / rel_path

        # Derive Package Name for Typo Distance
        pkg_name = Path(rel_path).parts[1] if len(Path(rel_path).parts) > 1 else "unknown"
        target_reference = TOP_NPM if ecosystem == "npm" else TOP_PYPI
        min_lev, max_jw = compute_typosquat_metrics(pkg_name, target_reference)

        extracted = {
            "uid": uid,
            "ecosystem": 1 if ecosystem == "npm" else 0,
            "label": label,
            "desc_len": 0,
            "has_readme": 0,
            "author_count": 0,
            "has_homepage": 0,
            "dep_count": 0,
            "dev_dep_count": 0,
            "has_preinstall": 0,
            "has_postinstall": 0,
            "has_cmdclass": 0,
            "min_levenshtein": min_lev,
            "max_jaro_winkler": round(max_jw, 4)
        }

        # 1. Parse NPM Ecosystem
        if ecosystem == "npm":
            target_file = pkg_full_path / "package.json"
            if target_file.exists():
                try:
                    with open(target_file, "r", encoding="utf-8", errors="ignore") as f:
                        extracted.update(parse_package_json(f.read()))
                except Exception:
                    pass

        # 2. Parse PyPI Ecosystem
        elif ecosystem == "pypi":
            setup_file = pkg_full_path / "setup.py"
            pkg_info_file = pkg_full_path / "PKG-INFO"

            # Parse AST from setup.py
            if setup_file.exists():
                try:
                    with open(setup_file, "r", encoding="utf-8", errors="ignore") as f:
                        ast_data = parse_setup_py_ast(f.read())
                        extracted.update(ast_data)
                except Exception:
                    pass

            # Augment ETM fields from PKG-INFO fallback if description is missing
            if pkg_info_file.exists() and extracted["desc_len"] == 0:
                try:
                    with open(pkg_info_file, "r", encoding="utf-8", errors="ignore") as f:
                        for line in f:
                            if line.startswith("Summary:"):
                                extracted["desc_len"] = len(line.replace("Summary:", "").strip())
                            elif line.startswith("Home-page:") or line.startswith("Project-URL:"):
                                extracted["has_homepage"] = 1
                            elif line.startswith("Author:") and extracted["author_count"] == 0:
                                extracted["author_count"] = 1
                except Exception:
                    pass

        feature_rows.append(extracted)

    df_features = pd.DataFrame(feature_rows)
    df_features.to_parquet(OUTPUT_FEATURES, engine="pyarrow", compression="snappy")
    print(f"[✓] Tier-1 metadata features extracted: {len(df_features)} records saved to {OUTPUT_FEATURES}")
    print(df_features.describe())

if __name__ == "__main__":

    extract_metadata_features()

[*] Extracting Tier-1 features across 5284 indexed packages...


<unknown>:127: SyntaxWarning: invalid escape sequence '\('
<unknown>:130: SyntaxWarning: invalid escape sequence '\['
<unknown>:138: SyntaxWarning: invalid escape sequence '\('
<unknown>:56: SyntaxWarning: invalid escape sequence '\('
<unknown>:57: SyntaxWarning: invalid escape sequence '\('
<unknown>:58: SyntaxWarning: invalid escape sequence '\)'
<unknown>:24: SyntaxWarning: invalid escape sequence '\.'
<unknown>:10: SyntaxWarning: invalid escape sequence '\s'


[✓] Tier-1 metadata features extracted: 5284 records saved to /content/drive/MyDrive/SHIELD_DATA/tier1_features.parquet
         ecosystem   label     desc_len  has_readme  author_count  \
count  5284.000000  5284.0  5284.000000  5284.00000   5284.000000   
mean      0.341976     0.0    33.054693     0.00265      0.432059   
std       0.474416     0.0   173.334075     0.05141      0.545941   
min       0.000000     0.0     0.000000     0.00000      0.000000   
25%       0.000000     0.0     0.000000     0.00000      0.000000   
50%       0.000000     0.0     0.000000     0.00000      0.000000   
75%       1.000000     0.0    45.000000     0.00000      1.000000   
max       1.000000     0.0  9141.000000     1.00000      7.000000   

       has_homepage    dep_count  dev_dep_count  has_preinstall  \
count   5284.000000  5284.000000    5284.000000     5284.000000   
mean       0.390424     1.514951       2.050530        0.001325   
std        0.487892    13.586967       6.116441        0.

In [ ]:
!apt-get update && apt-get install -y zstd && curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (5,147 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 38 not upgraded.
Need to g

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(3)  # Wait for daemon to bind to port 11434

# 3. Pull a fast, quantized model (Phi-3-Mini 3.8B or Llama-3-8B)
# Phi-3-mini (~2.3 GB) downloads in under 30s and runs extremely fast on T4
!ollama pull phi3:mini

In [ ]:
import os
import ast
import re
import json
import pandas as pd
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/SHIELD_DATA/raw_store")
MANIFEST_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest.parquet")
OUTPUT_CHUNKS_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/tier2_chunks.parquet")

# Excluded directory trees
IGNORED_DIRS = {"test", "tests", "docs", "doc", "fixtures", "example", "examples", "assets", "node_modules"}

# Sinks to monitor for Python
PY_SINKS = {
    "os": {"system", "popen", "spawn", "execv", "environ"},
    "subprocess": {"Popen", "run", "call", "check_output", "check_call"},
    "base64": {"b64decode", "standard_b64decode", "urlsafe_b64decode"},
    "socket": {"socket", "connect"},
    "builtins": {"eval", "exec", "__import__"}
}

# Sinks to monitor for JavaScript (Regex-based lexical extraction)
JS_SINK_PATTERNS = [
    re.compile(r"child_process\s*\.\s*(exec|spawn|execSync|fork)\s*\(", re.IGNORECASE),
    re.compile(r"require\s*\(\s*['\"]child_process['\"]\s*\)", re.IGNORECASE),
    re.compile(r"eval\s*\(", re.IGNORECASE),
    re.compile(r"new\s+Function\s*\(", re.IGNORECASE),
    re.compile(r"Buffer\s*\.\s*from\s*\([^)]*['\"]base64['\"]\)", re.IGNORECASE),
    re.compile(r"net\s*\.\s*(connect|createConnection)\s*\(", re.IGNORECASE),
    re.compile(r"process\s*\.\s*env", re.IGNORECASE),
    re.compile(r"(http|https)\s*\.\s*(get|request)\s*\(", re.IGNORECASE)
]

def sanitize_python_code(code_str: str) -> str:
    """Removes comments and docstrings from Python code."""
    try:
        parsed = ast.parse(code_str)
        for node in ast.walk(parsed):
            if isinstance(node, (ast.FunctionDef, ast.ClassDef, ast.Module)):
                if (node.body and isinstance(node.body[0], ast.Expr) and
                        isinstance(node.body[0].value, ast.Constant) and
                        isinstance(node.body[0].value.value, str)):
                    node.body.pop(0)
        return ast.unparse(parsed)
    except Exception:
        # Fallback to regex comment stripping
        return re.sub(r"#.*", "", code_str)

def sanitize_js_code(code_str: str) -> str:
    """Removes single-line and multi-line comments from JavaScript."""
    pattern = r"(\".*?\"|\'.*?\')|(/\*.*?\*/|//[^\r\n]*$)"
    regex = re.compile(pattern, re.MULTILINE | re.DOTALL)
    def replacer(match):
        return match.group(1) if match.group(1) else ""
    clean = regex.sub(replacer, code_str)
    return "\n".join([line.strip() for line in clean.splitlines() if line.strip()])

def extract_python_sinks(file_path: Path, max_context_lines=3) -> list:
    """Uses AST to locate call nodes matching sensitive APIs and extracts surrounding code lines."""
    findings = []
    try:
        raw_code = file_path.read_text(encoding="utf-8", errors="ignore")
        clean_code = sanitize_python_code(raw_code)
        lines = clean_code.splitlines()
        tree = ast.parse(clean_code)

        for node in ast.walk(tree):
            sink_detected = None
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Attribute):
                    val_id = getattr(node.func.value, "id", None)
                    attr_name = node.func.attr
                    if val_id in PY_SINKS and attr_name in PY_SINKS[val_id]:
                        sink_detected = f"{val_id}.{attr_name}"
                elif isinstance(node.func, ast.Name):
                    if node.func.id in PY_SINKS["builtins"]:
                        sink_detected = node.func.id

            if sink_detected and hasattr(node, "lineno"):
                line_idx = node.lineno - 1
                start = max(0, line_idx - max_context_lines)
                end = min(len(lines), line_idx + max_context_lines + 1)
                context_slice = "\n".join(lines[start:end])
                findings.append({
                    "file": file_path.name,
                    "sink": sink_detected,
                    "line": node.lineno,
                    "context": context_slice
                })
    except Exception:
        pass
    return findings

def extract_js_sinks(file_path: Path, max_context_lines=3) -> list:
    """Scans JavaScript source files for pattern-matched sink signatures."""
    findings = []
    try:
        raw_code = file_path.read_text(encoding="utf-8", errors="ignore")
        clean_code = sanitize_js_code(raw_code)
        lines = clean_code.splitlines()

        for idx, line in enumerate(lines):
            for pat in JS_SINK_PATTERNS:
                if pat.search(line):
                    start = max(0, idx - max_context_lines)
                    end = min(len(lines), idx + max_context_lines + 1)
                    findings.append({
                        "file": file_path.name,
                        "sink": pat.pattern[:25],
                        "line": idx + 1,
                        "context": "\n".join(lines[start:end])
                    })
                    break
    except Exception:
        pass
    return findings

def process_package_chunks(pkg_dir: Path, ecosystem: str) -> list:
    """Scans eligible package files, excluding documentation and test directories."""
    all_chunks = []
    for root, dirs, files in os.walk(pkg_dir):
        dirs[:] = [d for d in dirs if d.lower() not in IGNORED_DIRS]
        for f in files:
            file_path = Path(root) / f
            if ecosystem == "pypi" and f.endswith(".py"):
                chunks = extract_python_sinks(file_path)
                all_chunks.extend(chunks)
            elif ecosystem == "npm" and f.endswith(".js"):
                chunks = extract_js_sinks(file_path)
                all_chunks.extend(chunks)
    return all_chunks

def build_tier2_dataset():
    df_manifest = pd.read_parquet(MANIFEST_PATH)
    print(f"[*] Building Tier-2 AST Context Payloads across {len(df_manifest)} packages...")

    records = []
    for _, row in df_manifest.iterrows():
        uid = row["uid"]
        ecosystem = row["ecosystem"]
        label = row["label"]
        rel_path = row["package_rel_path"]
        pkg_full_path = DATA_ROOT / rel_path

        chunks = []
        if pkg_full_path.exists():
            chunks = process_package_chunks(pkg_full_path, ecosystem)

        # Build collapsed LLM prompt content
        if chunks:
            chunk_text = "\n---\n".join([f"[{c['file']}:{c['line']} - Sink: {c['sink']}]\n{c['context']}" for c in chunks])
        else:
            chunk_text = "NO_HIGH_RISK_SINKS_DETECTED"

        records.append({
            "uid": uid,
            "ecosystem": ecosystem,
            "label": label,
            "sink_count": len(chunks),
            "isolated_code_payload": chunk_text
        })

    out_df = pd.DataFrame(records)
    out_df.to_parquet(OUTPUT_CHUNKS_PATH, engine="pyarrow", compression="snappy")
    print(f"[✓] Tier-2 chunks compiled: {len(out_df)} records saved to {OUTPUT_CHUNKS_PATH}")
    print(out_df["sink_count"].describe())

if __name__ == "__main__":
    build_tier2_dataset()

[*] Building Tier-2 AST Context Payloads across 5284 packages...


<unknown>:127: SyntaxWarning: invalid escape sequence '\('
<unknown>:130: SyntaxWarning: invalid escape sequence '\['
<unknown>:138: SyntaxWarning: invalid escape sequence '\('
<unknown>:56: SyntaxWarning: invalid escape sequence '\('
<unknown>:57: SyntaxWarning: invalid escape sequence '\('
<unknown>:58: SyntaxWarning: invalid escape sequence '\)'
<unknown>:24: SyntaxWarning: invalid escape sequence '\.'
<unknown>:10: SyntaxWarning: invalid escape sequence '\s'
<unknown>:188: SyntaxWarning: invalid escape sequence '\d'


[✓] Tier-2 chunks compiled: 5284 records saved to /content/drive/MyDrive/SHIELD_DATA/tier2_chunks.parquet
count    5284.000000
mean        0.275360
std         2.756451
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max       109.000000
Name: sink_count, dtype: float64


In [ ]:
import json
import requests

OLLAMA_ENDPOINT = "http://localhost:11434/api/generate"

SYSTEM_PROMPT = """You are S.H.I.E.L.D., an automated supply-chain application security auditor.
Analyze the following context-minimized AST code chunks extracted from an installer package.
Determine if the code performs:
1. Unauthorized data exfiltration (network sockets, HTTP requests during installation).
2. Environment variable or secret harvesting (~/.ssh, ~/.aws, os.environ, process.env).
3. Arbitrary remote command execution or reverse shell spawning.
4. Payload obfuscation (base64 decoding passed into exec/eval).

Return your verdict STRICTLY as a single-line JSON object adhering to this schema:
{
  "threat_flags": ["exfiltration", "obfuscation", "rce", "credential_theft"],
  "llm_score": <float between 0.0 for SAFE and 1.0 for MALICIOUS>,
  "rationale": "<concise explanation in 20 words or less>"
}"""

def query_ollama_auditor(code_payload: str, model="llama3:8b") -> dict:
    if code_payload == "NO_HIGH_RISK_SINKS_DETECTED":
        return {
            "threat_flags": [],
            "llm_score": 0.05,
            "rationale": "No sensitive system execution or exfiltration sinks detected."
        }

    prompt = f"PACKAGE CODE CHUNKS TO AUDIT:\n{code_payload}\n\nJSON RESPONSE:"
    payload = {
        "model": model,
        "system": SYSTEM_PROMPT,
        "prompt": prompt,
        "stream": False,
        "format": "json"
    }

    try:
        resp = requests.post(OLLAMA_ENDPOINT, json=payload, timeout=30)
        if resp.status_code == 200:
            return json.loads(resp.json()["response"])
    except Exception as e:
        return {"error": str(e), "llm_score": 0.5, "rationale": "Inference failed"}
    return {"llm_score": 0.5, "rationale": "Empty response"}

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from pathlib import Path

# Paths mapped to the Colab Google Drive mount
MANIFEST_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest.parquet")
FEATURES_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/tier1_features.parquet")

def benchmark_dataset():
    print("=" * 60)
    print("S.H.I.E.L.D. DATASET BENCHMARK & QUALITY AUDIT")
    print("=" * 60)

    if not MANIFEST_PATH.exists():
        print("[!] Manifest not found. Ensure Day 4 compilation completed.")
        return

    df = pd.read_parquet(MANIFEST_PATH)
    total = len(df)
    malicious = (df['label'] == 1).sum()
    benign = (df['label'] == 0).sum()

    print(f"Total Unique Samples : {total}")
    print(f"Malicious Samples    : {malicious} ({(malicious/total)*100:.2f}%)")
    print(f"Benign Samples       : {benign} ({(benign/total)*100:.2f}%)")

    print("\nSource Collection Breakdown:")
    print(df['source_collection'].value_counts())

    zero_size = (df['size_bytes'] == 0).sum()
    if zero_size > 0:
        print(f"\n[!] Alert: {zero_size} records have 0 bytes and represent corrupted archives.")
    else:
        print("\n[✓] Integrity check passed. Zero empty archives detected.")
    print("=" * 60)

def evaluate_tier1_baseline():
    print("\n[*] Initializing Tier-1 XGBoost Baseline Evaluation...")
    if not FEATURES_PATH.exists():
        print("[!] Features file not found. Ensure Day 5 extraction completed.")
        return

    df_features = pd.read_parquet(FEATURES_PATH)

    # Drop identifiers, keeping only the computed ETM, DTM, and Lexical features
    X = df_features.drop(columns=["uid", "label", "ecosystem"], errors="ignore")
    y = df_features["label"]

    # Initialize 5-Fold Stratified split to preserve class ratios
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=42
    )

    acc_scores, recall_scores, fpr_scores = [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)

        # Calculate False Positive Rate (FP / (FP + TN))
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        acc_scores.append(acc)
        recall_scores.append(recall)
        fpr_scores.append(fpr)

    print(f"\n[✓] 5-Fold Stratified CV Results:")
    print(f"Accuracy : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Recall   : {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
    print(f"FPR      : {np.mean(fpr_scores):.4f} ± {np.std(fpr_scores):.4f}")

if __name__ == "__main__":
    benchmark_dataset()
    evaluate_tier1_baseline()

S.H.I.E.L.D. DATASET BENCHMARK & QUALITY AUDIT
Total Unique Samples : 5284
Malicious Samples    : 0 (0.00%)
Benign Samples       : 5284 (100.00%)

Source Collection Breakdown:
source_collection
pypi    3579
npm     1705
Name: count, dtype: int64

[✓] Integrity check passed. Zero empty archives detected.

[*] Initializing Tier-1 XGBoost Baseline Evaluation...


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricW


[✓] 5-Fold Stratified CV Results:
Accuracy : 1.0000 ± 0.0000
Recall   : 0.0000 ± 0.0000
FPR      : 0.0000 ± 0.0000


In [ ]:
import pandas as pd
import hashlib
from pathlib import Path

# --- File Paths ---
BENIGN_FEATURES_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/tier1_features.parquet")
MALICIOUS_NPM_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest_pypi_malicious_combined.csv")
MALICIOUS_PYPI_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest_npm_malicious_combined.csv")
FINAL_FEATURES_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/tier1_features_combined.parquet")

def standardize_malicious_df(df, is_npm=True):
    """Maps the CSV column names to the expected Day 5 schema."""

    # Strip whitespace from column names to prevent KeyErrors
    df.columns = df.columns.str.strip()

    # Create a deterministic UID for these records
    df["uid"] = df["package_name"].apply(lambda x: hashlib.sha256(f"malicious_{x}".encode()).hexdigest()[:16])

    # 1 for npm, 0 for pypi
    df["ecosystem"] = 1 if is_npm else 0
    df["label"] = 1  # Ensure all are marked malicious

    # Map ETM Features
    df["desc_len"] = df.get("etm_description_length", pd.Series(0, index=df.index)).fillna(0)
    df["has_readme"] = df.get("etm_has_readme", pd.Series(0, index=df.index)).fillna(0)
    df["has_homepage"] = df.get("etm_has_homepage", pd.Series(0, index=df.index)).fillna(0)

    author_col = "etm_maintainer_count" if is_npm else "etm_author_count"
    df["author_count"] = df.get(author_col, pd.Series(0, index=df.index)).fillna(0)

    # Map DTM Features
    dep_col = "dtm_max_dependency_count" if is_npm else "dtm_requires_dist_count"
    df["dep_count"] = df.get(dep_col, pd.Series(0, index=df.index)).fillna(0)
    df["dev_dep_count"] = 0 # Not explicitly tracked in CSV, default to 0

    # Map Hooks
    if is_npm:
        df["has_preinstall"] = df.get("dtm_has_lifecycle_hook", pd.Series(0, index=df.index)).fillna(0)
        df["has_postinstall"] = df.get("dtm_has_lifecycle_hook", pd.Series(0, index=df.index)).fillna(0)
        df["has_cmdclass"] = 0
    else:
        df["has_preinstall"] = 0
        df["has_postinstall"] = 0
        df["has_cmdclass"] = 1 # Approximation for malicious PyPI

    # Map Lexical
    df["min_levenshtein"] = df.get("typo_min_distance_to_top2000", pd.Series(999, index=df.index)).fillna(999)
    df["max_jaro_winkler"] = df.get("is_likely_typosquat", pd.Series(0, index=df.index)).fillna(0).apply(lambda x: 0.9 if x == 1 else 0.0)

    # Keep only the target columns
    expected_cols = [
        "uid", "ecosystem", "label", "desc_len", "has_readme", "author_count",
        "has_homepage", "dep_count", "dev_dep_count", "has_preinstall",
        "has_postinstall", "has_cmdclass", "min_levenshtein", "max_jaro_winkler"
    ]

    return df[expected_cols]

def merge_datasets():
    print("[*] Loading benign features...")
    df_benign = pd.read_parquet(BENIGN_FEATURES_PATH)

    # Filter just in case any 1s slipped in previously
    df_benign = df_benign[df_benign["label"] == 0]

    print("[*] Loading and standardizing malicious CSVs...")
    df_mal_npm = pd.read_csv(MALICIOUS_NPM_PATH)
    df_mal_pypi = pd.read_csv(MALICIOUS_PYPI_PATH)

    std_mal_npm = standardize_malicious_df(df_mal_npm, is_npm=True)
    std_mal_pypi = standardize_malicious_df(df_mal_pypi, is_npm=False)

    print("[*] Merging datasets...")
    df_final = pd.concat([df_benign, std_mal_npm, std_mal_pypi], ignore_index=True)

    # Fill any missing values resulting from the merge
    df_final = df_final.fillna(0)

    # Save the unified file
    df_final.to_parquet(FINAL_FEATURES_PATH, engine="pyarrow", compression="snappy")

    print("=" * 60)
    print("FINAL TIER-1 DATASET DISTRIBUTION")
    print("=" * 60)
    print(f"Total Packages : {len(df_final)}")
    print(f"Benign (0)     : {(df_final['label'] == 0).sum()}")
    print(f"Malicious (1)  : {(df_final['label'] == 1).sum()}")
    print("=" * 60)
    print(f"[✓] Saved to: {FINAL_FEATURES_PATH}")

if __name__ == "__main__":
    merge_datasets()

[*] Loading benign features...
[*] Loading and standardizing malicious CSVs...
[*] Merging datasets...
FINAL TIER-1 DATASET DISTRIBUTION
Total Packages : 8305
Benign (0)     : 5284
Malicious (1)  : 3021
[✓] Saved to: /content/drive/MyDrive/SHIELD_DATA/tier1_features_combined.parquet


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from pathlib import Path

# --- Configuration ---
# Pointing to the newly merged dataset from the previous step
FEATURES_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/tier1_features_combined.parquet")

def audit_dataset(df):
    print("=" * 60)
    print("S.H.I.E.L.D. DATASET QUALITY AUDIT")
    print("=" * 60)

    total = len(df)
    malicious = (df['label'] == 1).sum()
    benign = (df['label'] == 0).sum()

    print(f"Total Unique Samples : {total}")
    print(f"Benign Samples (0)   : {benign} ({(benign/total)*100:.2f}%)")
    print(f"Malicious Samples (1): {malicious} ({(malicious/total)*100:.2f}%)")

    print("\nEcosystem Breakdown:")
    # ecosystem: 1 for npm, 0 for pypi
    print(df.groupby(['ecosystem', 'label']).size().unstack(fill_value=0).rename(index={1: 'npm', 0: 'pypi'}, columns={0: 'Benign', 1: 'Malicious'}))
    print("=" * 60)

def evaluate_tier1_baseline(df):
    print("\n[*] Initializing Tier-1 XGBoost Baseline Evaluation...")

    # Drop identifiers and labels to isolate the feature matrix (X)
    X = df.drop(columns=["uid", "label", "ecosystem"], errors="ignore")
    y = df["label"]

    # Initialize 5-Fold Stratified split to preserve the benign/malicious ratio in every fold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Initialize the XGBoost Classifier
    model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric='logloss',
        random_state=42,
        use_label_encoder=False
    )

    acc_scores, recall_scores, fpr_scores = [], [], []

    print("[*] Running Cross-Validation (This may take a moment)...")
    fold = 1
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Train the model
        model.fit(X_train, y_train)

        # Predict on the test fold
        y_pred = model.predict(X_test)

        # Calculate metrics
        acc = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)

        # Confusion matrix to calculate False Positive Rate (FP / (FP + TN))
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

        acc_scores.append(acc)
        recall_scores.append(recall)
        fpr_scores.append(fpr)
        print(f"  -> Fold {fold} completed.")
        fold += 1

    print("\n" + "=" * 60)
    print("FINAL TIER-1 TABULAR BASELINE RESULTS (5-Fold CV)")
    print("=" * 60)
    print(f"Accuracy : {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"Recall   : {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
    print(f"FPR      : {np.mean(fpr_scores):.4f} ± {np.std(fpr_scores):.4f}")
    print("=" * 60)

if __name__ == "__main__":
    if not FEATURES_PATH.exists():
        print(f"[!] Cannot find dataset at {FEATURES_PATH}")
        print("Please ensure the merge script from the previous step completed successfully.")
    else:
        # Load the combined dataset
        df_combined = pd.read_parquet(FEATURES_PATH)

        # Execute the pipeline
        audit_dataset(df_combined)
        evaluate_tier1_baseline(df_combined)

S.H.I.E.L.D. DATASET QUALITY AUDIT
Total Unique Samples : 8305
Benign Samples (0)   : 5284 (63.62%)
Malicious Samples (1): 3021 (36.38%)

Ecosystem Breakdown:
label      Benign  Malicious
ecosystem                   
pypi         3477       2833
npm          1807        188

[*] Initializing Tier-1 XGBoost Baseline Evaluation...
[*] Running Cross-Validation (This may take a moment)...
  -> Fold 1 completed.


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [09:42:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [09:42:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  -> Fold 2 completed.
  -> Fold 3 completed.


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [09:42:35] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [09:42:35] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  -> Fold 4 completed.


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [09:42:36] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  -> Fold 5 completed.

FINAL TIER-1 TABULAR BASELINE RESULTS (5-Fold CV)
Accuracy : 1.0000 ± 0.0000
Recall   : 1.0000 ± 0.0000
FPR      : 0.0000 ± 0.0000


In [4]:
import pandas as pd
import hashlib
from pathlib import Path

# --- File Paths ---
BENIGN_FEATURES_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/tier1_features.parquet")
MALICIOUS_NPM_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest_pypi_malicious_combined.csv")
MALICIOUS_PYPI_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/manifest_npm_malicious_combined.csv")
FINAL_FEATURES_PATH = Path("/content/drive/MyDrive/SHIELD_DATA/tier1_features_combined.parquet")

def merge_leak_free():
    print("[*] Loading benign features...")
    df_benign = pd.read_parquet(BENIGN_FEATURES_PATH)
    df_benign = df_benign[df_benign["label"] == 0]

    # Combine benign npm hooks to match malicious CSV's single hook column
    df_benign["has_hook"] = df_benign.apply(
        lambda row: 1 if (row.get("has_preinstall", 0) == 1 or row.get("has_postinstall", 0) == 1) and row["ecosystem"] == 1 else 0,
        axis=1
    )

    # Restrict benign data to the strict intersection
    benign_safe = df_benign[[
        "uid", "ecosystem", "label", "desc_len", "has_readme",
        "has_homepage", "author_count", "dep_count", "has_hook"
    ]].copy()

    print("[*] Formatting Malicious NPM CSV...")
    df_mal_npm = pd.read_csv(MALICIOUS_NPM_PATH)
    df_mal_npm.columns = df_mal_npm.columns.str.strip()

    npm_safe = pd.DataFrame()
    npm_safe["uid"] = df_mal_npm["package_name"].apply(lambda x: hashlib.sha256(f"mal_npm_{x}".encode()).hexdigest()[:16])
    npm_safe["ecosystem"] = 1  # 1 for NPM
    npm_safe["label"] = 1
    npm_safe["desc_len"] = df_mal_npm.get("etm_description_length", pd.Series(0, index=df_mal_npm.index)).fillna(0)
    npm_safe["has_readme"] = df_mal_npm.get("etm_has_readme", pd.Series(0, index=df_mal_npm.index)).fillna(0)
    npm_safe["has_homepage"] = df_mal_npm.get("etm_has_homepage", pd.Series(0, index=df_mal_npm.index)).fillna(0)
    npm_safe["author_count"] = df_mal_npm.get("etm_maintainer_count", pd.Series(0, index=df_mal_npm.index)).fillna(0)
    npm_safe["dep_count"] = df_mal_npm.get("dtm_max_dependency_count", pd.Series(0, index=df_mal_npm.index)).fillna(0)
    npm_safe["has_hook"] = df_mal_npm.get("dtm_has_lifecycle_hook", pd.Series(0, index=df_mal_npm.index)).fillna(0)

    print("[*] Formatting Malicious PyPI CSV...")
    df_mal_pypi = pd.read_csv(MALICIOUS_PYPI_PATH)
    df_mal_pypi.columns = df_mal_pypi.columns.str.strip()

    pypi_safe = pd.DataFrame()
    pypi_safe["uid"] = df_mal_pypi["package_name"].apply(lambda x: hashlib.sha256(f"mal_pypi_{x}".encode()).hexdigest()[:16])
    pypi_safe["ecosystem"] = 0  # 0 for PyPI
    pypi_safe["label"] = 1
    pypi_safe["desc_len"] = df_mal_pypi.get("etm_description_length", pd.Series(0, index=df_mal_pypi.index)).fillna(0)
    pypi_safe["has_readme"] = df_mal_pypi.get("etm_has_readme", pd.Series(0, index=df_mal_pypi.index)).fillna(0)
    pypi_safe["has_homepage"] = df_mal_pypi.get("etm_has_homepage", pd.Series(0, index=df_mal_pypi.index)).fillna(0)
    pypi_safe["author_count"] = df_mal_pypi.get("etm_author_count", pd.Series(0, index=df_mal_pypi.index)).fillna(0)
    pypi_safe["dep_count"] = df_mal_pypi.get("dtm_requires_dist_count", pd.Series(0, index=df_mal_pypi.index)).fillna(0)
    pypi_safe["has_hook"] = 0  # PyPI malicious CSV lacks this, so we force 0 to prevent leakage

    print("[*] Merging leak-free datasets...")
    df_final = pd.concat([benign_safe, npm_safe, pypi_safe], ignore_index=True)
    df_final = df_final.fillna(0)

    # Save the unified file
    df_final.to_parquet(FINAL_FEATURES_PATH, engine="pyarrow", compression="snappy")

    print("=" * 60)
    print("LEAK-FREE TIER-1 DATASET DISTRIBUTION")
    print("=" * 60)
    print(f"Total Packages : {len(df_final)}")
    print(f"Benign (0)     : {(df_final['label'] == 0).sum()}")
    print(f"Malicious (1)  : {(df_final['label'] == 1).sum()}")
    print(f"Features used  : {list(benign_safe.columns)}")
    print("=" * 60)

if __name__ == "__main__":
    merge_leak_free()

[*] Loading benign features...
[*] Formatting Malicious NPM CSV...
[*] Formatting Malicious PyPI CSV...
[*] Merging leak-free datasets...
LEAK-FREE TIER-1 DATASET DISTRIBUTION
Total Packages : 8305
Benign (0)     : 5284
Malicious (1)  : 3021
Features used  : ['uid', 'ecosystem', 'label', 'desc_len', 'has_readme', 'has_homepage', 'author_count', 'dep_count', 'has_hook']
